In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from preprocessing import get_features_and_target
from visualizer import plot_visualizer
import plotly.graph_objects as go
from tabpfn import TabPFNRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
import torch
from torch import nn
from torch import device

In [2]:
import huggingface_hub
huggingface_hub.login()

# Getting Dataframe

In [3]:
# Load the training and development datasets
train_df = pd.read_csv("data/train_data.csv")
dev_df = pd.read_csv("data/test_data.csv")
sc = StandardScaler()

target_column = "PullTest (N)" 

x_train, y_train = get_features_and_target(train_df, target_column)
x_dev, y_dev = get_features_and_target(dev_df, target_column)

x_train_scale = sc.fit_transform(X=x_train)
x_dev_scale = sc.transform(x_dev)


In [4]:
train_df.shape[0]

276

# Defining Model

In [17]:
model_name = 'XGBoost'
#model_name = 'RandomForest'
#model_name = 'TabPFN'
#model_name = 'Transformer'

# Fit Model

In [19]:
if model_name == 'TabPFN':

    # Initialize the regressor
    regressor = TabPFNRegressor()  # Uses TabPFN-2.5 weights, trained on synthetic data only.
    # To use TabPFN v2:
    # regressor = TabPFNRegressor.create_default_for_version(ModelVersion.V2)
    regressor.fit(x_train_scale, y_train)

    # Predict on the test set
    predictions = regressor.predict(x_dev_scale)

elif model_name == 'XGBoost':

    # Convert the data into DMatrix format
    dtrain = xgb.DMatrix(x_train_scale, label=y_train)
    dtest = xgb.DMatrix(x_dev_scale, label=y_dev)

    # Set the parameters for the XGBoost model
    params = {
        'objective': 'reg:squarederror',
        'max_depth': 1,
        'eta': 0.57,
        'eval_metric': 'rmse',
    }

    # Train the model
    num_boost_round = 13
    bst = xgb.train(params, dtrain, num_boost_round)

    # Make predictions
    predictions = bst.predict(dtest)

elif model_name == 'RandomForest':

    # Set the parameters for the Random Forest model
    params = {
            'n_estimators': 19,
            'max_depth': 5,
            'min_samples_split': 6,
            'min_samples_leaf': 3,
            'random_state': 42,
    }

    predictions = RandomForestRegressor(**params).fit(x_train_scale, y_train).predict(x_dev_scale)

elif model_name == 'Transformer':

    from torch.utils.data import TensorDataset, DataLoader

    class TabularTransformer(nn.Module):
        def __init__(self, num_features, d_model=64, nhead=4, num_layers=4):
            super().__init__()

            self.input_proj = nn.Linear(num_features, d_model)

            encoder_layer = nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=nhead,
                batch_first=True
            )
            self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

            self.head = nn.Linear(d_model, 1)

        def forward(self, x):   
            x = self.input_proj(x)          # [batch, d_model]
            x = x.unsqueeze(1)              # [batch, seq=1, d_model]
            x = self.encoder(x)             # [batch, 1, d_model]
            x = x.squeeze(1)
            x = self.head(x)       # [batch, 1]
            return x

    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    model = TabularTransformer(num_features=x_train.shape[1]).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)
    y_dev_tensor = torch.tensor(y_dev, dtype=torch.float32).unsqueeze(1).to(device)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).to(device)
    X_train = torch.tensor(x_train_scale, dtype=torch.float32).to(device)
    X_dev = torch.tensor(x_dev_scale, dtype=torch.float32).to(device)


    batch_size = 32

    train_dataset = TensorDataset(X_train, y_train_tensor)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)


    class RMSELoss(nn.Module):
        def __init__(self):
            super().__init__()
            self.mse = nn.MSELoss()

        def forward(self, pred, target):
            return torch.sqrt(self.mse(pred, target))

    
    criterion = RMSELoss()

    best_val_loss = float("inf")
    best_model_path = "best_tabular_transformer.pt"

    for epoch in range(1000):

        # -------------------------
        # TRAINING
        # -------------------------
        model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()
        

        # -------------------------
        # VALIDATION
        # -------------------------
        model.eval()
        with torch.no_grad():
            predictions = model(X_dev)
            val_loss = criterion(predictions, y_dev_tensor)   # y_dev_tensor explained below

        # -------------------------
        # LOGGING
        # -------------------------
        if epoch % 100 == 0:
            print(f"Epoch {epoch:4d} | Train RMSE: {loss.item():.4f} | "
                f"Val RMSE: {val_loss.item():.4f}")

        # -------------------------
        # SAVE BEST MODEL
        # -------------------------
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), best_model_path)


# Check Validation Data

In [20]:
# Category array (must be aligned with y_dev)
categories = dev_df.groupby("Sample ID")["Category"].first().values

plot_visualizer(
    true_vals=y_dev,
    pred_vals=predictions,
    categories=categories,
    title=f"Validation Samples: True vs Prediction ({model_name}) by Category - Data-Driven Training"
)

# Check Validation Loss and R2

In [21]:

# Calculate MAE and RMSE and R2
mae  = mean_absolute_error(y_dev, predictions)
rmse = np.sqrt(root_mean_squared_error(y_dev, predictions))**2
R2   = r2_score(y_dev, predictions)



print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2: {R2:.2f}")


MAE:  147.00
RMSE: 247.14
R2: 0.64


# Cross Validation

In [22]:
cross_df = pd.read_csv("data/train_dev_data.csv")

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

mae_list = []
rmse_list = []
R2_list = []

for fold, (train_index, val_index) in enumerate(skf.split(cross_df["Sample ID"], cross_df["Category"])):
    x_tr, y_tr = get_features_and_target(cross_df.iloc[train_index], target_column) 
    x_val, y_val = get_features_and_target(cross_df.iloc[val_index], target_column)

    x_tr_scale = sc.fit_transform(X=x_tr)
    x_val_scale = sc.transform(x_val)

    if model_name == 'XGBoost':
        dtrain = xgb.DMatrix(x_tr_scale, label=y_tr)
        dval = xgb.DMatrix(x_val_scale, label=y_val)

        params = {
            'objective': 'reg:squarederror',
            'max_depth': 1,
            'eta': 0.57,
            'eval_metric': 'rmse'
        }

        num_boost_round = 20
        bst = xgb.train(params, dtrain, num_boost_round)

        preds = bst.predict(dval)

    elif model_name == 'RandomForest':

        params = {
            'n_estimators': 11,
            'max_depth': 5,
            'min_samples_split': 2,
            'min_samples_leaf': 4,
            'random_state': 42,
        }

        preds = RandomForestRegressor(**params).fit(x_tr_scale, y_tr).predict(x_val_scale)
        
    elif model_name == 'TabPFN':
        regressor = TabPFNRegressor()
        regressor.fit(x_tr, y_tr)

        preds = regressor.predict(x_val)

    elif model_name == 'Transformer':

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        X_val_tensor = torch.tensor(x_val_scale, dtype=torch.float32).to(device)
        # 1. Recreate the model architecture
        model = TabularTransformer(num_features=x_train.shape[1]).to(device)

        # 2. Load the saved weights
        model.load_state_dict(torch.load("best_tabular_transformer_datadriven_fpull.pt", map_location=device))
        model.eval()

        # 3. Prepare validation fold
        X_val_tensor = torch.tensor(x_val_scale, dtype=torch.float32).to(device)

        # 4. Predict
        with torch.no_grad():
            preds = model(X_val_tensor).cpu().numpy().ravel()


    mae  = mean_absolute_error(y_val, preds)
    rmse = root_mean_squared_error(y_val, preds)
    R2   = r2_score(y_val, preds)

    mae_list.append(mae) 
    rmse_list.append(rmse) 
    R2_list.append(R2)

    # Categories
    categories= cross_df.iloc[val_index]["Category"].values

    plot_visualizer(
        true_vals=y_val,
        pred_vals=preds,
        categories=categories,
        title=f"Fold {fold+1}: True vs Prediction ({model_name}) by Category - Cross-Validation Data-Driven Training"
    )

    print(f"\nFold {fold+1}")
    print("MAE :", mae)
    print("RMSE:", rmse)
    print("R²  :", R2)

mae_mean = np.mean(mae_list) 
rmse_mean = np.mean(rmse_list) 
R2_mean = np.mean(R2_list) 



Fold 1
MAE : 138.89556061039127
RMSE: 206.32812414378716
R²  : 0.707251162513721



Fold 2
MAE : 147.93646786077235
RMSE: 252.1962419958166
R²  : 0.7127199883454697



Fold 3
MAE : 112.76811245553863
RMSE: 159.82352429289986
R²  : 0.6595455413220844


In [46]:
print(f"mean MAE:  {mae_mean:.2f} ± {np.std(mae_list):.2f}")
print(f"mean RMSE: {rmse_mean:.2f} ± {np.std(rmse_list):.2f}")
print(f"mean R²:   {R2_mean:.2f} ± {np.std(R2_list):.2f}")

mean MAE:  133.84 ± 1.19
mean RMSE: 237.89 ± 18.33
mean R²:   0.66 ± 0.03


In [52]:
print(f"mean MAE:  {mae_mean:.2f} ± {np.std(mae_list):.2f}")
print(f"mean RMSE: {rmse_mean:.2f} ± {np.std(rmse_list):.2f}")
print(f"mean R²:   {R2_mean:.2f} ± {np.std(R2_list):.2f}")


mean MAE:  84.84 ± 11.48
mean RMSE: 154.37 ± 29.09
mean R²:   0.86 ± 0.03


In [23]:
print(f"mean MAE:  {mae_mean:.2f} ± {np.std(mae_list):.2f}")
print(f"mean RMSE: {rmse_mean:.2f} ± {np.std(rmse_list):.2f}")
print(f"mean R²:   {R2_mean:.2f} ± {np.std(R2_list):.2f}")

mean MAE:  133.20 ± 14.91
mean RMSE: 206.12 ± 37.71
mean R²:   0.69 ± 0.02


# Select between XGB and RandomForest

In [ ]:
#model_type = "xgbregressor"
model_type = "random_forest" 

# Model Creation

In [ ]:
def create_model(model_type, params):
    if model_type == "xgbregressor":
        return XGBRegressor(
            objective='reg:squarederror',
            max_depth=params['max_depth'],
            learning_rate=params['learning_rate'],
            n_estimators=params['n_estimators'],
            eval_metric='rmse'
        )
    elif model_type == "random_forest":
        return RandomForestRegressor(
            n_estimators=params['n_estimators'],
            max_depth=params['max_depth'],
            min_samples_split=params['min_samples_split'],
            min_samples_leaf=params['min_samples_leaf'],
            random_state=42,
        )


# Parameter Selection

In [ ]:
if model_type == "xgbregressor":
    param_grid = []
    max_depth_values = range(1, 5)
    eta_values = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7] # first search [0.5, 0.51, 0.52, 0.53, 0.54, 0.55, 0.56, 0.57]
    n_estimators_values = range(50, 201, 50)

    for md in max_depth_values:
        for eta in eta_values:
            for ne in n_estimators_values:
                param_grid.append({
                    'max_depth': md,
                    'learning_rate': eta,
                    'n_estimators': ne
                })

elif model_type == "random_forest":
    param_grid = []
    n_estimators_values = [17] # first search range(100, 1001, 100) #range(10, 101, 10)
    max_depth_values = [11]
    min_samples_split_values = [2] # first search range(2, 11)
    min_samples_leaf_values = [5]

    for ne in n_estimators_values:
        for md in max_depth_values:
            for mss in min_samples_split_values:
                for msl in min_samples_leaf_values:
                    param_grid.append({
                        'n_estimators': ne,
                        'max_depth': md,
                        'min_samples_split': mss,
                        'min_samples_leaf': msl
                    })


# Gridsearch

In [ ]:
best_rmse = float("inf")
best_params = None
best_model = None

for params in param_grid:
    print(f"\nTesting params: {params}")

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

    mae_list = []
    rmse_list = []
    R2_list = []

    for fold, (train_index, val_index) in enumerate(
            skf.split(cross_df["Sample ID"], cross_df["Category"])):

        X_tr, y_tr = get_features_and_target(cross_df.iloc[train_index], target_column)
        X_val, y_val = get_features_and_target(cross_df.iloc[val_index], target_column)

        X_tr_scale = sc.fit_transform(X_tr)
        X_val_scale = sc.transform(X_val)

        model = create_model(model_type, params)
        
        model.fit(X_tr_scale, y_tr)
        preds = model.predict(X_val_scale)

        mae = mean_absolute_error(y_val, preds) 
        rmse = root_mean_squared_error(y_val, preds) 
        R2 = r2_score(y_val, preds) 

        mae_list.append(mae) 
        rmse_list.append(rmse) 
        R2_list.append(R2)

    mae_mean = np.mean(mae_list) 
    rmse_mean = np.mean(rmse_list) 
    R2_mean = np.mean(R2_list) 

    if rmse_mean < best_rmse:
        best_rmse = rmse_mean
        best_params = params
        best_model = model

        print("\n==============================")
        print(" BEST MODEL FOUND ")
        print("==============================")
        print(f"Best RMSE:   {best_rmse:.2f}")
        print(f"Best Params: {best_params}")



Testing params: {'n_estimators': 17, 'max_depth': 11, 'min_samples_split': 2, 'min_samples_leaf': 5}

 BEST MODEL FOUND 
Best RMSE:   254.09
Best Params: {'n_estimators': 17, 'max_depth': 11, 'min_samples_split': 2, 'min_samples_leaf': 5}


# RandomSearch

In [ ]:
from scipy.stats import randint
from sklearn.model_selection import RandomizedSearchCV

# use a different variable name to avoid overwriting the notebook-level 'model' string
rf_estimator = RandomForestRegressor()

param_dist = {
    'n_estimators': randint(10, 200),
    'max_depth': randint(1, 20),
    'min_samples_split': randint(2, 11),
    'min_samples_leaf': randint(1, 11)
}

randomized_search = RandomizedSearchCV(
    rf_estimator,
    param_distributions=param_dist,
    n_iter=10000,
    cv=5,
    scoring='neg_mean_squared_error',  # regression scoring
    n_jobs=-1,
    random_state=42
)

# fit on the existing training variables (use scaled features if desired)
randomized_search.fit(x_train_scale, y_train)

,estimator,RandomForestRegressor()
,param_distributions,"{'max_depth': <scipy.stats....x7a329b303b80>, 'min_samples_leaf': <scipy.stats....x7a32a0485060>, 'min_samples_split': <scipy.stats....x7a329b8cd900>, 'n_estimators': <scipy.stats....x7a32b84f9a20>}"
,n_iter,10000
,scoring,'neg_mean_squared_error'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [10]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

model_name = "rf"  # or "xgbregressor"

def create_regressor(model_name, params):
    if model_name == "rf":
        return RandomForestRegressor(**params, random_state=42)
    elif model_name == "xgb":
        return XGBRegressor(
            **params,
            random_state=42,
            tree_method="hist",
            eval_metric="rmse"
        )
    else:
        raise ValueError(f"Unknown model: {model_name}")



In [11]:
from scipy.stats import randint, uniform

param_spaces = {
    "rf": {
        "n_estimators": randint(1, 50),
        "max_depth": randint(1, 20),
        "min_samples_split": randint(2, 10),
        "min_samples_leaf": randint(1, 10)
    },
    "xgb": {
          # Erster Durchlauf: randint(50, 300)
        'max_depth' : randint(1, 5),
        'learning_rate': randint(50, 71),  #Erster durchlauf: randint(1, 11)
        'n_estimators' : randint(1, 30),  

    }
}

In [12]:
def sample_params(space):
    params = {}
    for k, v in space.items():
        val = v.rvs()

        # Convert learning_rate integer → stepped float
        if k == "learning_rate":
            val = round(val / 100.0, 2)   # erster durchlauf: val = round(val / 10.0, 1) 1→0.1, 2→0.2, ..., 10→1.0

        params[k] = val
    return params


In [13]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np

def randomized_cv_search(model_name, cross_df, target_column, n_iter):
    space = param_spaces[model_name]

    best_rmse = float("inf")
    best_params = None
    best_model = None

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

    for i in range(n_iter):
        params = sample_params(space)
        print(f"\nTesting params: {params}")

        mae_list, rmse_list, r2_list = [], [], []

        for fold, (train_index, val_index) in enumerate(
                skf.split(cross_df["Sample ID"], cross_df["Category"])):

            train_df = cross_df.iloc[train_index].copy()
            val_df   = cross_df.iloc[val_index].copy()

            X_tr, y_tr = get_features_and_target(train_df, target_column)
            X_val, y_val = get_features_and_target(val_df, target_column)

            X_tr_scale = sc.fit_transform(X_tr)
            X_val_scale = sc.transform(X_val)

            model = create_regressor(model_name, params)
            model.fit(X_tr_scale, y_tr)

            preds = model.predict(X_val_scale)

            mae = mean_absolute_error(y_val, preds)
            rmse = np.sqrt(np.mean((y_val - preds)**2))
            r2 = r2_score(y_val, preds)

            mae_list.append(mae)
            rmse_list.append(rmse)
            r2_list.append(r2)

        rmse_mean = np.mean(rmse_list)
        print(f"RMSE={rmse_mean:.3f}")

        if rmse_mean < best_rmse:
            best_rmse = rmse_mean
            best_params = params
            best_model = model

    return best_model, best_params, best_rmse


In [15]:
best_model, best_params, best_rmse = randomized_cv_search(
    model_name="rf",         # ← SWITCH TO XGBOOST
    cross_df=cross_df,
    target_column="PullTest (N)",
    n_iter=10000
)



Testing params: {'n_estimators': 17, 'max_depth': 16, 'min_samples_split': 2, 'min_samples_leaf': 6}
RMSE=252.468

Testing params: {'n_estimators': 22, 'max_depth': 5, 'min_samples_split': 5, 'min_samples_leaf': 1}
RMSE=217.903

Testing params: {'n_estimators': 32, 'max_depth': 10, 'min_samples_split': 6, 'min_samples_leaf': 7}
RMSE=251.179

Testing params: {'n_estimators': 13, 'max_depth': 14, 'min_samples_split': 5, 'min_samples_leaf': 1}
RMSE=224.175

Testing params: {'n_estimators': 19, 'max_depth': 11, 'min_samples_split': 4, 'min_samples_leaf': 4}
RMSE=229.994

Testing params: {'n_estimators': 36, 'max_depth': 16, 'min_samples_split': 2, 'min_samples_leaf': 6}
RMSE=247.625

Testing params: {'n_estimators': 4, 'max_depth': 11, 'min_samples_split': 3, 'min_samples_leaf': 3}
RMSE=237.838

Testing params: {'n_estimators': 29, 'max_depth': 9, 'min_samples_split': 9, 'min_samples_leaf': 3}
RMSE=220.316

Testing params: {'n_estimators': 3, 'max_depth': 17, 'min_samples_split': 6, 'min_

In [ ]:
print("\n==============================")
print(" BEST MODEL FOUND ")
print("==============================")
print(f"Best RMSE:   {best_rmse:.2f}")
print(f"Best Params: {best_params}")

#==============================
# BEST MODEL FOUND XGBoost
#==============================
#ohne explode
#Best RMSE:   206.12
#Best Params: {'max_depth': 1, 'learning_rate': 0.57, 'n_estimators': 20}

#Best RMSE:   241.12
#Best Params: {'n_estimators': 13, 'max_depth': 1, 'learning_rate': 0.57}
#Best RMSE:   241.86
#Best Params: {'n_estimators': 15, 'max_depth': 1, 'learning_rate': 0.6}
#Best RMSE:   245.36
#Best Params: {'n_estimators': 50, 'max_depth': 1, 'learning_rate': 0.2, }
#Best RMSE:   257.09
#Best Params: {'n_estimators': 28, 'max_depth': 4, 'learning_rate': np.float64(0.10519775309146498)}
#Best RMSE:   263.07
#Best Params: {'n_estimators': 29, 'max_depth': 1, 'learning_rate': 0.5478342030460285}

#==============================
# BEST MODEL FOUND RandomForest
#==============================
#ohne explode
#Best Params: {'n_estimators': 48, 'max_depth': 7, 'min_samples_split': 2, 'min_samples_leaf': 1}
#Best RMSE:   206.77

#Best Params: {'n_estimators': 11, 'max_depth': 5, 'min_samples_split': 2, 'min_samples_leaf': 4}
#Best RMSE:   229.32
#Best Params: {'n_estimators': 19, 'max_depth': 5, 'min_samples_split': 6, 'min_samples_leaf': 3}
#Best RMSE:   230.34
#Best Params: {'n_estimators': 19, 'max_depth': 7, 'min_samples_split': 9, 'min_samples_leaf': 3}
#Best RMSE:   230.57
#Best Params: {'n_estimators': 20, 'max_depth': 6, 'min_samples_split': 10, 'min_samples_leaf': 3}
#Best RMSE:   234.09
#Best Params: {'n_estimators': 100, 'max_depth': 10, 'min_samples_split': 7, 'min_samples_leaf': 3}
#Best RMSE:   250.05
#Best Params: {'n_estimators': 20, 'max_depth': 11, 'min_samples_split': 5, 'min_samples_leaf': 5}
#Best RMSE:   254.09
#Best Params: {'n_estimators': 17, 'max_depth': 11, 'min_samples_split': 2, 'min_samples_leaf': 5}


 BEST MODEL FOUND 
Best RMSE:   206.77
Best Params: {'n_estimators': 48, 'max_depth': 7, 'min_samples_split': 2, 'min_samples_leaf': 1}
